# 🩺 Diabetes Risk Prediction — Explainable ML System

**Dataset:** CDC BRFSS 2015 Diabetes Health Indicators (253,680 records · 21 features)  
**Author:** Ankit Kumar  
**Internship:** Academy of Skill Development, Kolkata — Data Science, AI & ML (May–July 2025)

---

## Project Highlights
| Component | Detail |
|---|---|
| Dataset | CDC BRFSS 2015 — 253,680 real survey records |
| Models | Logistic Regression, Random Forest, XGBoost, LightGBM, SVM + Stacking Ensemble |
| Class Imbalance | SMOTE oversampling |
| Explainability | SHAP (global summary + local waterfall plots) |
| Evaluation | Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix |
| Feature Engineering | 3 clinically-motivated composite features |
| Deployment | Streamlit web app |

---

## Workflow
1. Environment Setup
2. Data Loading & Exploration (EDA)
3. Data Preprocessing & Feature Engineering
4. Train/Test Split + SMOTE + Scaling
5. Model Training (5 models)
6. Model Evaluation & Comparison
7. Stacking Ensemble
8. SHAP Explainability
9. Final Results Summary
10. Model Saving

## 1. Environment Setup

In [ ]:
import sys, os
# Add src to path so we can import our modules
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import seaborn as sns

# Our custom modules
from preprocessing    import load_data, check_missing, engineer_features, split_and_scale, FEATURE_DESCRIPTIONS
from models           import get_models, get_stacking_ensemble, train_model, evaluate_model, cross_validate_model, save_model
from visualizations   import (plot_target_distribution, plot_feature_distributions,
                               plot_correlation_heatmap, plot_boxplots_by_class,
                               plot_model_comparison, plot_roc_curves,
                               plot_confusion_matrices, plot_feature_importance_rf)
from explainability   import (get_shap_explainer, compute_shap_values,
                               plot_shap_summary, plot_shap_bar,
                               plot_shap_waterfall, get_top_features)

print('All imports successful ✓')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
import sklearn; print(f'sklearn : {sklearn.__version__}')
import xgboost; print(f'XGBoost : {xgboost.__version__}')
import lightgbm; print(f'LightGBM: {lightgbm.__version__}')
import shap; print(f'SHAP    : {shap.__version__}')

## 2. Data Loading & Exploration

In [ ]:
DATA_PATH = '../data/diabetes_binary_health_indicators_BRFSS2015.csv'
df = load_data(DATA_PATH)

print('\n--- First 5 rows ---')
df.head()

In [ ]:
print('--- Dataset Info ---')
print(f'Shape         : {df.shape}')
print(f'Total records : {df.shape[0]:,}')
print(f'Features      : {df.shape[1] - 1}')
print(f'\nData types:')
print(df.dtypes)
print(f'\nBasic statistics:')
df.describe().round(2)

In [ ]:
# Feature reference card
print('--- Feature Descriptions ---')
for feat, desc in FEATURE_DESCRIPTIONS.items():
    print(f'  {feat:<30}: {desc}')

In [ ]:
# Missing values check
check_missing(df)

In [ ]:
# Class distribution plot
plot_target_distribution(df)
print('Plot saved to outputs/plots/01_target_distribution.png')

In [ ]:
# Feature distributions by class
from preprocessing import FEATURE_COLS
plot_feature_distributions(df, FEATURE_COLS)
print('Plot saved to outputs/plots/02_feature_distributions.png')

In [ ]:
# Correlation heatmap
plot_correlation_heatmap(df)
print('Plot saved to outputs/plots/03_correlation_heatmap.png')

In [ ]:
# Boxplots for continuous features
continuous = ['BMI', 'MentHlth', 'PhysHlth', 'Age', 'GenHlth', 'Education', 'Income']
plot_boxplots_by_class(df, continuous)
print('Plot saved to outputs/plots/04_boxplots_by_class.png')

## 3. Feature Engineering

We add three clinically-motivated composite features:

| Feature | Formula | Rationale |
|---|---|---|
| `MetabolicRisk` | Weighted sum of BMI, blood pressure, cholesterol, heart disease, stroke | Captures overall cardiometabolic risk |
| `LifestyleScore` | Physical activity + fruit/veg intake − smoking − heavy alcohol | Healthy lifestyle proxy |
| `HealthcareBurden` | Mental health days + physical health days | Overall health impairment |

These are **not in the original dataset** — they are our engineered additions that give the model richer signal.

In [ ]:
df_eng = engineer_features(df)
print('Engineered features added:')
print(df_eng[['MetabolicRisk', 'LifestyleScore', 'HealthcareBurden', 'Diabetes_binary']].describe().round(2))

## 4. Train/Test Split + SMOTE + Scaling

- **Split:** 80% train / 20% test (stratified)
- **SMOTE:** Oversample the minority class (Diabetic) in the training set only — never on test data
- **Scaling:** StandardScaler fit on training data, applied to both

In [ ]:
data = split_and_scale(df_eng, test_size=0.20, random_state=42, apply_smote=True)

X_train        = data['X_train']
X_test         = data['X_test']
y_train        = data['y_train']
y_test         = data['y_test']
X_train_scaled = data['X_train_scaled']
X_test_scaled  = data['X_test_scaled']
scaler         = data['scaler']
feature_names  = data['feature_names']

print(f'\nFinal shapes:')
print(f'  X_train_scaled : {X_train_scaled.shape}')
print(f'  X_test_scaled  : {X_test_scaled.shape}')
print(f'  Feature names  : {feature_names}')

## 5. Model Training

Training five models:
1. **Logistic Regression** — baseline linear classifier
2. **Random Forest** — ensemble of decision trees (Bagging)
3. **XGBoost** — extreme gradient boosting (sequential trees)
4. **LightGBM** — faster gradient boosting (leaf-wise growth)
5. **SVM** — maximum-margin classifier with RBF kernel

**Note:** Logistic Regression and SVM use scaled data. Tree models use raw data (they are scale-invariant).

In [ ]:
models = get_models()
trained_models = {}

# Tree-based models: use unscaled data
TREE_MODELS = ['Random Forest', 'XGBoost', 'LightGBM']

for name, model in models.items():
    if name in TREE_MODELS:
        X_tr, X_te = X_train, X_test
    else:
        X_tr, X_te = X_train_scaled, X_test_scaled
    
    trained_model = train_model(model, X_tr, y_train, model_name=name)
    trained_models[name] = (trained_model, X_te)

print('\nAll models trained ✓')

## 6. Model Evaluation

In [ ]:
results = []
models_for_plots = {}   # name → model (all using test set)

for name, (model, X_te) in trained_models.items():
    metrics = evaluate_model(model, X_te, y_test, model_name=name)
    results.append(metrics)
    models_for_plots[name] = model

results_df = pd.DataFrame(results)
print('\n--- Summary Table ---')
results_df.set_index('Model')

In [ ]:
# 5-Fold Cross-Validation for robust comparison
print('--- Cross-Validation (ROC-AUC) ---')
for name, (model, X_te) in trained_models.items():
    if name in TREE_MODELS:
        X_cv = X_train
    else:
        X_cv = X_train_scaled
    cross_validate_model(model, X_cv, y_train, model_name=name)

In [ ]:
# Visual comparison
plot_model_comparison(results_df)

# ROC curves — need models on their correct test sets
# Build a wrapper dict where each model has predict_proba access
# We'll use scaled data for all for the ROC plot (SVM, LR need it)
# For tree models this is fine too — StandardScaler on tree data is neutral
plot_roc_curves(
    {name: model for name, (model, _) in trained_models.items()},
    X_test_scaled, y_test
)

plot_confusion_matrices(
    {name: model for name, (model, _) in trained_models.items()},
    X_test_scaled, y_test
)

print('Plots saved to outputs/plots/')

In [ ]:
# Random Forest feature importance (Gini)
rf_model = trained_models['Random Forest'][0]
plot_feature_importance_rf(rf_model, feature_names)

# Best model so far
best_row = results_df.loc[results_df['ROC-AUC'].idxmax()]
print(f'\n🏆 Best individual model: {best_row["Model"]} (ROC-AUC = {best_row["ROC-AUC"]})')

## 7. Stacking Ensemble

**Architecture:**
- Level 0 (Base Learners): Random Forest + XGBoost + LightGBM
- Level 1 (Meta-Learner): Logistic Regression

The base learners make out-of-fold predictions during 5-fold CV. These predictions become features for the meta-learner. This lets the meta-learner learn *when to trust* each base model.

In [ ]:
print('Building Stacking Ensemble ...')
stack = get_stacking_ensemble()
stack = train_model(stack, X_train, y_train, model_name='Stacking Ensemble')

stack_metrics = evaluate_model(stack, X_test, y_test, model_name='Stacking Ensemble')
results.append(stack_metrics)
results_df = pd.DataFrame(results)
trained_models['Stacking Ensemble'] = (stack, X_test)

In [ ]:
print('--- Final Results — All Models ---')
results_df.sort_values('ROC-AUC', ascending=False).set_index('Model').style.highlight_max(
    subset=['Accuracy','Precision','Recall','F1-Score','ROC-AUC'],
    color='#90EE90'
)

## 8. SHAP Explainability

SHAP (SHapley Additive exPlanations) explains *why* the model made each prediction.

- **Summary plot:** Which features have the most impact overall?
- **Bar plot:** Clean ranking of feature importance
- **Waterfall plot:** Explains a single individual prediction

We use XGBoost for SHAP (TreeExplainer is fast and exact for tree models).

In [ ]:
print('Computing SHAP values for XGBoost ...')
xgb_model  = trained_models['XGBoost'][0]

explainer  = get_shap_explainer(xgb_model, X_train, 'XGBoost')

# Use a sample of 2000 test points for speed
sample_idx = np.random.default_rng(42).choice(len(X_test), size=2000, replace=False)
X_sample   = X_test[sample_idx]

shap_values = compute_shap_values(explainer, X_sample)
print('SHAP values computed ✓')

In [ ]:
# Global summary (beeswarm)
plot_shap_summary(shap_values, X_sample, feature_names, 'XGBoost',
                  save_dir='../outputs/plots')

# Global bar chart
plot_shap_bar(shap_values, feature_names, 'XGBoost',
              save_dir='../outputs/plots')

print('SHAP global plots saved.')

In [ ]:
# Local explanation: explain prediction for a single patient
# Pick a diabetic patient from the test sample for a more interesting waterfall
y_sample = y_test[sample_idx]
diabetic_indices = np.where(y_sample == 1)[0]
patient_idx = int(diabetic_indices[0])   # first diabetic patient in our sample

print(f'Explaining prediction for patient #{patient_idx} (Actual: Diabetic)')
print(f'Model prediction probability: {xgb_model.predict_proba(X_sample[[patient_idx]])[:, 1][0]:.3f}')

plot_shap_waterfall(shap_values, patient_idx, feature_names, 'XGBoost',
                    save_dir='../outputs/plots')

In [ ]:
# Top features table
print('--- Top 10 Features by Mean |SHAP| Value ---')
top_df = get_top_features(shap_values, feature_names, top_n=10)
top_df

## 9. Final Results Summary

In [ ]:
print('=' * 70)
print('  DIABETES RISK PREDICTION — FINAL RESULTS SUMMARY')
print('=' * 70)
print(f'  Dataset   : CDC BRFSS 2015 Diabetes Health Indicators')
print(f'  Records   : 253,680')
print(f'  Features  : 21 original + 3 engineered = 24 total')
print(f'  Split     : 80% train / 20% test (stratified)')
print(f'  Imbalance : SMOTE applied on training set')
print(f'  Scaling   : StandardScaler')
print()
print(results_df.sort_values('ROC-AUC', ascending=False).to_string(index=False))
print()

best = results_df.loc[results_df['ROC-AUC'].idxmax()]
print(f'  🏆 Best Model : {best["Model"]}')
print(f'     Accuracy  : {best["Accuracy"]:.4f}')
print(f'     ROC-AUC   : {best["ROC-AUC"]:.4f}')
print(f'     F1-Score  : {best["F1-Score"]:.4f}')
print('=' * 70)

## 10. Save Models

In [ ]:
import joblib, os

os.makedirs('../outputs/models', exist_ok=True)

# Save all trained models
for name, (model, _) in trained_models.items():
    fname = name.replace(' ', '_').lower() + '.pkl'
    joblib.dump(model, f'../outputs/models/{fname}')
    print(f'Saved: {fname}')

# Save scaler separately (needed for Streamlit app)
joblib.dump(scaler, '../outputs/models/scaler.pkl')
print('Saved: scaler.pkl')

# Save feature names
import json
with open('../outputs/models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)
print('Saved: feature_names.json')

print('\nAll models saved ✓')